# BACnet — devices, objects, priorities and COV

IoTCom's own `BacnetDevice` runs an air handling unit; the client is `BacnetIpClient`, an adapter over ela-compil's BACnet library. They share an in-memory network where broadcasts use a multicast group. On a real network drop `UseInMemory` and both use UDP 47808.

## Setup
> Working from a clone? Run `dotnet pack -c Release -o artifacts/packages` at the repo root and add
> `#i "nuget: <repo>/artifacts/packages"` before the `#r` lines.

In [ ]:
#r "nuget: IoTCom.Net, 0.25.0-preview.1"
#r "nuget: IoTCom.Net.Native.Modbus, 0.25.0-preview.1"
#r "nuget: IoTCom.Net.Adapters.Bacnet, 0.25.0-preview.1"

## Objects and the priority array
A commandable object's present value is its highest-priority command; relinquishing hands it back.

In [ ]:
using IoTCom.Net;
using IoTCom.Net.Adapters.Bacnet;
using IoTCom.Net.Protocols.Bacnet;
using IoTCom.Net.Transports;
using System.Net;

var fan = BacnetObject.BinaryOutput(1, "Supply fan", relinquishDefault: true, "Start", "Stop");
fan.Write(new(BacnetPropertyId.PresentValue), [BacnetValue.Enumerated(0)], 8);    // operator stops it
fan.Write(new(BacnetPropertyId.PresentValue), [BacnetValue.Enumerated(1)], 16);   // a schedule wants it on
Console.WriteLine($"{fan.FormatValue()} at priority {fan.ActivePriority}");
fan.Write(new(BacnetPropertyId.PresentValue), [BacnetValue.Null], 8);
Console.WriteLine($"{fan.FormatValue()} at priority {fan.ActivePriority}");
Console.WriteLine(Convert.ToHexString(BacnetServices.ReadProperty(BacnetObjectId.Parse("AI1"), new(BacnetPropertyId.PresentValue))));

## Discover and read
Who-Is finds the unit; ReadPropertyMultiple reads names and values in one request per object.

In [ ]:
var bnNet = new InMemoryDatagramNetwork();
var bnGroup = new IPEndPoint(IPAddress.Parse("239.255.186.192"), 47808);
var ahu = BacnetAhuSimulator.Create(o => { o.UseInMemory(bnNet, new IPEndPoint(IPAddress.Parse("10.60.0.2"), 47808)); o.BroadcastAddress = bnGroup; });
await ahu.StartAsync();
var bms = BacnetIpClient.Create(o => { o.UseInMemory(bnNet, new IPEndPoint(IPAddress.Parse("10.60.0.20"), 47808)); o.BroadcastAddress = bnGroup; });
await bms.ConnectAsync();
foreach (var d in await bms.DiscoverAsync(TimeSpan.FromMilliseconds(300))) Console.WriteLine(d);
foreach (var obj in (await bms.ReadObjectListAsync(2604)).Where(o => o.Type != BacnetObjectType.Device))
{
    var r = await bms.ReadPropertiesAsync(2604, obj, [BacnetPropertyId.ObjectName, BacnetPropertyId.PresentValue]);
    Console.WriteLine($"{obj.ShortName,-5} {r[0].Values![0],-30} {r[1].Values![0]}");
}

## Follow a point by COV
The device notifies when the supply temperature moves by its COV increment (0.2 °C), and once right after subscribing.

In [ ]:
var cov = new List<string>();
var sub = await bms.SubscribeCovAsync(2604, BacnetObjectId.Parse("AI1"), n => { lock (cov) cov.Add(n.Values[0].Values[0].ToString()); });
for (var i = 0; i < 6; i++) { ahu.Step(10); await Task.Delay(100); }
await sub.DisposeAsync();
lock (cov) Console.WriteLine("AI1: " + string.Join(" -> ", cov));

## Commands are opt-in
This client is read-only; a second one with AllowWrites() stops the fan at priority 8 and relinquishes it.

In [ ]:
try { await bms.WritePropertyAsync(2604, BacnetObjectId.Parse("BO1"), BacnetPropertyId.PresentValue, BacnetValue.Enumerated(0), 8); } catch (ReadOnlyModeException e) { Console.WriteLine(e.Message); }
var operatorBms = BacnetIpClient.Create(o => { o.UseInMemory(bnNet, new IPEndPoint(IPAddress.Parse("10.60.0.21"), 47808)); o.BroadcastAddress = bnGroup; o.AllowWrites(); });
await operatorBms.ConnectAsync();
await operatorBms.DiscoverAsync(TimeSpan.FromMilliseconds(300));
await operatorBms.WritePropertyAsync(2604, BacnetObjectId.Parse("BO1"), BacnetPropertyId.PresentValue, BacnetValue.Enumerated(0), 8);
var slots = await operatorBms.ReadPropertyAsync(2604, BacnetObjectId.Parse("BO1"), BacnetPropertyId.PriorityArray);
Console.WriteLine(string.Join(" ", slots.Select((v, i) => v.Tag == BacnetApplicationTag.Null ? "." : $"{i + 1}:{v}")));
await operatorBms.WritePropertyAsync(2604, BacnetObjectId.Parse("BO1"), BacnetPropertyId.PresentValue, BacnetValue.Null, 8);
Console.WriteLine($"fan command {ahu.FanCommand.FormatValue()}");
await operatorBms.DisposeAsync();
await bms.DisposeAsync();
await ahu.DisposeAsync();

## Going further
`iotcom bacnet demo`, `iotcom bacnet discover`, `iotcom bacnet objects 2604 --sim`, `iotcom bacnet serve`, `iotcom bacnet decode <hex>`, the Gallery's *Air handling unit over BACnet* and the BacnetBms sample. See `docs/en/protocols/bacnet.md`.

*IoTCom.Net — built by Gravicode Studios, led by Kang Fadhil.*